# L04: Build a modelling-ready feature set

**Day 1.** A stratified split, imputation that keeps the signal in missingness, encoding and scaling inside a Pipeline fitted on training rows only, and PCA from covariance to a defended component count.

- Reads: checkpoints/L02_clean.csv
- Writes: work/L04_split.csv, work/L04_feature_spec.json

**Before you start.** Open this notebook from the **Notebooks** folder of the class workspace, next to the **data** and **checkpoints** folders, and run the first two cells.

**How this lab runs.** Work through the cells in order on your own. Each one says what to do with it: **RUN** supplied code, **PREDICT** before running, **DECIDE** by filling a blank, or **WRITE** an answer in words. When most of the room is done, we discuss the answers as a class, then walk through the worked solution together.

### Step 1 · RUN

In [ ]:
# RUN · setup: finds the class files and checks that today's are there. Run it first, every time.
from pathlib import Path
# The class files sit in the folder holding "data" and "checkpoints": this notebook's folder or the one above it.
# (On a local install, a folder named AIDSA in your user folder also works.)
_here = Path.cwd()
ROOT = next((p for p in [_here, *_here.parents, Path.home() / "AIDSA"] if (p / "data").is_dir() and (p / "checkpoints").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Class files not found: open this notebook from the Notebooks folder of the class workspace.")
DATA, CHECKPOINTS, WORK = ROOT / "data", ROOT / "checkpoints", ROOT / "work"
NEEDED = ["checkpoints/L02_clean.csv"]
missing = [name for name in NEEDED if not (ROOT / name).exists()]
if missing:
    raise FileNotFoundError("Not in the workspace yet: " + ", ".join(missing) + ". Ask your instructor: today's files are not in the workspace yet.")
WORK.mkdir(exist_ok=True)
SOURCE = "checkpoint"  # change to "mine" to use your own saved work where it exists
SEED = 2026
import numpy as np
rng = np.random.default_rng(SEED)

def input_path(name):
    """Where this lab reads an earlier lab's output: your own work folder if you chose "mine" and the file is there,
    otherwise the supplied checkpoint."""
    mine = WORK / name
    return mine if SOURCE == "mine" and mine.exists() else CHECKPOINTS / name

print("Found the class files in:", ROOT, "| start from:", SOURCE)

### Step 2 · RUN

In [ ]:
# RUN · helper functions: the tested course helpers this lab uses. Run it once, after the setup cell.
# You do not need to read them to do the lab; each is explained where the lab uses it.
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


NUMERIC = ["aircraft_age_years", "total_flight_hours", "days_since_inspection", "ground_time_hours",
           "cargo_weight_klb", "temperature_c", "sortie_duration_hours", "open_major", "open_minor"]


CATEGORICAL = ["aircraft_type", "home_base", "mission_type"]


TARGET = "maint_delay"


def make_split(df, target=TARGET, test_size=0.25, seed=2026, id_col="mission_id"):
    """Stratified train and test split, returned as a table of identifiers and their split.

    Stratifying on the target keeps the roughly 70 to 30 balance identical in both parts.

    Lab L04.
    """
    train_ids, test_ids = train_test_split(df[id_col], test_size=test_size, stratify=df[target],
                                           random_state=seed)
    split = pd.concat([pd.DataFrame({id_col: train_ids, "split": "train"}),
                       pd.DataFrame({id_col: test_ids, "split": "test"})])
    return split.sort_values(id_col).reset_index(drop=True)


def build_preprocessor(numeric=NUMERIC, categorical=CATEGORICAL):
    """Unfitted preprocessing: numeric median imputation with missing indicators, then scaling;
    categorical one-hot encoding that ignores unseen levels.

    Fit it on training rows only.

    Lab L04.
    """
    numeric_steps = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                              ("scale", StandardScaler())])
    return ColumnTransformer([
        ("num", numeric_steps, numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
    ])


def pca_summary(Z):
    """Eigen-decomposition of the covariance of standardised data Z, largest component first.

    Returns a table with each component's eigenvalue, share of variance and cumulative share.

    Lab L04.
    """
    Z = np.asarray(Z, dtype=float)
    eigenvalues = np.linalg.eigvalsh(np.cov(Z, rowvar=False))[::-1]
    share = eigenvalues / eigenvalues.sum()
    return pd.DataFrame({"component": np.arange(1, len(share) + 1), "eigenvalue": eigenvalues,
                         "share": share, "cumulative": np.cumsum(share)})

print("Helper functions ready")

## Part 1: Split

**Specification.** Set aside a test set before anything is learned from the data, keeping the delay rate identical
in both parts.

### Step 3 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline

df = pd.read_csv(input_path("L02_clean.csv"), encoding="utf-8")
split = make_split(df, target=TARGET, test_size=0.25, seed=SEED)
df = df.merge(split, on="mission_id", validate="one_to_one")
train, test = df[df["split"] == "train"], df[df["split"] == "test"]

X_cols = NUMERIC + CATEGORICAL          # identifiers, dates and flags are not features
X_train, y_train = train[X_cols], train[TARGET]
X_test, y_test = test[X_cols], test[TARGET]
X_all, y_all = df[X_cols], df[TARGET]            # every row, both halves: kept only for comparison
print(f"train {len(train):,} rows, delay rate {y_train.mean():.3f} | test {len(test):,} rows, delay rate {y_test.mean():.3f}")
print("features:", X_cols)

### Step 4 · WRITE

**WRITE · your interpretation**. Why is the test set set aside now, before imputation, scaling or PCA, and why stratify on the outcome?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 2: Missing data and encoding

**Specification.** Look at where cargo weight is missing, then assemble the preprocessing: median imputation with a
missing indicator and scaling for numeric columns, one-hot encoding for categories.

### Step 5 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
missing_by_outcome = df.groupby("maint_delay")["cargo_weight_klb"].apply(lambda s: s.isna().mean())
print("share of cargo weight missing: on time {:.1%}, delayed {:.1%}".format(*missing_by_outcome))

# Supplied: numeric columns are imputed with the median plus a 0/1 "was missing" column, then scaled;
# categorical columns are one-hot encoded, ignoring any level not seen in training.
preprocessor = build_preprocessor(NUMERIC, CATEGORICAL)
preprocessor

### Step 6 · WRITE

**WRITE · your interpretation**. Cargo is missing four times as often on delayed departures. What does that do to a model if you
simply fill the gaps with the median, and why does the missing indicator fix it?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 3: Train-only fitting

**Specification.** Put the preprocessing and a logistic regression into one Pipeline and fit it on training rows only.
Then compare it with preprocessing that was fitted on all rows.

### Step 7 · PREDICT

**PREDICT · before you run the next cell**. If the preprocessing is fitted on all 3,000 rows instead of the 2,250 training rows, will the test
score go up, down, or stay the same? Why?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 8 · DECIDE

In [ ]:
# DECIDE · DP-L04-1 · which rows the fit call receives.
# Replace ____ with your choice. The choices are taught on the slide "The pipeline: the object that enforces the order".
pipeline = Pipeline([("prep", preprocessor), ("model", LogisticRegression(max_iter=1000))])
pipeline.fit(____)
auc = roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1])
print(f"test AUC of the pipeline, fitted on the rows you chose: {auc:.3f}")

### Step 9 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
# The mistake, for comparison: preprocessing fitted on every row, test rows included.
leaky_prep = build_preprocessor(NUMERIC, CATEGORICAL).fit(df[X_cols])
leaky_model = LogisticRegression(max_iter=1000).fit(leaky_prep.transform(X_train), y_train)
leaky_auc = roc_auc_score(y_test, leaky_model.predict_proba(leaky_prep.transform(X_test))[:, 1])
learned = lambda prep: prep.named_transformers_["num"].named_steps["impute"].statistics_[
    NUMERIC.index("cargo_weight_klb")]
print(f"test AUC, preprocessing fitted on all rows: {leaky_auc:.3f}")
print(f"cargo median learned: {learned(pipeline.named_steps['prep']):.1f} (train only) "
      f"against {learned(leaky_prep):.1f} (all rows)")

### Step 10 · WRITE

**WRITE · your interpretation**. Why does the Pipeline make train-only fitting the default rather than something you must remember?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 4: PCA

**Specification.** Decompose the covariance of the standardised numeric training features into principal components,
and choose how many to keep.

### Step 11 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
# Standardise with the training-only fitted preprocessing, keeping the nine numeric features.
prep = pipeline.named_steps["prep"]
Z = prep.transform(X_train)[:, :len(NUMERIC)]
pca = pca_summary(Z)                    # eigenvalues of the covariance matrix, largest first
print(pca.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(pca["component"], pca["eigenvalue"], marker="o", label="eigenvalue (scree)")
ax.axhline(1.0, linestyle=":", label="eigenvalue 1")
ax2 = ax.twinx()
ax2.plot(pca["component"], pca["cumulative"], marker="s", color="grey", label="cumulative share")
ax2.axhline(0.9, linestyle="--", color="grey")
ax.set_xlabel("component")
ax.set_ylabel("eigenvalue")
ax2.set_ylabel("cumulative share of variance")
fig.legend(loc="center right")
plt.show()

### Step 12 · DECIDE

In [ ]:
# DECIDE · DP-L04-2 · how many components to keep.
# Replace ____ with your choice. The choices are taught on the slide "How many components to keep: two rules of thumb".
# Read the cumulative variance and the scree plot above, then choose.
N_COMPONENTS = ____
kept = pca.loc[N_COMPONENTS - 1, "cumulative"]
print(f"keeping {N_COMPONENTS} of {len(pca)} components retains {kept:.1%} of the variance")

### Step 13 · WRITE

**WRITE · your interpretation**. Defend your component count against both criteria. What does the last component's tiny eigenvalue
tell you about the features?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 14 · RUN

In [ ]:
# RUN · saves this part's output to work/.
split.to_csv(WORK / "L04_split.csv", index=False, encoding="utf-8")
spec = {"numeric": NUMERIC, "categorical": CATEGORICAL, "target": TARGET,
        "n_components": N_COMPONENTS, "baseline_test_auc": round(auc, 4),
        "transformed_features": list(prep.get_feature_names_out())}
(WORK / "L04_feature_spec.json").write_text(json.dumps(spec, indent=2), encoding="utf-8")
print("saved work/L04_split.csv and work/L04_feature_spec.json")